# [M3] XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
## Buổi S03: Thiết Kế Kiến Trúc Mạng Nơ-ron với torch.nn.Module

---

### Metadata Khóa Học
* **Module:** M3 — XÂY MODEL AI SÂU HƠN BẰNG PYTORCH
* **Buổi:** S03
* **Thời lượng dự kiến:** 120 phút
* **Sản phẩm đầu ra (Session Output):** Notebook thiết kế kiến trúc mạng nơ-ron MLP chuẩn mực
* **Mức độ liên quan VOAI / Olympic AI:** Khả năng xây dựng kiến trúc mạng nơ-ron tùy biến cho mọi bài toán Olympic AI.

#### Mục tiêu học tập (Learning Objectives)
- Hiểu kiến trúc Perceptron Đa tầng (Multi-Layer Perceptron - MLP) và nguyên lý xâu chuỗi các tầng tuyến tính.
- Kế thừa và làm chủ lớp trừu tượng `torch.nn.Module` để đóng gói kiến trúc mạng nơ-ron.
- Hiểu bản chất toán học và sự cần thiết của các Hàm kích hoạt phi tuyến (Non-linear Activations): ReLU, Sigmoid, LeakyReLU.
- Kiểm tra, đếm số lượng tham số học được (`parameters()`) và truy vết kích thước Tensor qua từng tầng mạng.

#### Điều kiện tiên quyết (Prerequisites)
- Hoàn thành Buổi M3_S01 và M3_S02.
- Hiểu phép nhân ma trận giữa tensor đầu vào và ma trận trọng số tầng ẩn.

---


## 1. WHY? — TẠI SAO PHẢI HỌC CHỦ ĐỀ NÀY?


Nếu chúng ta chỉ xếp chồng nhiều tầng tuyến tính liên tiếp nhau: $\hat{y} = W_2 (W_1 X + b_1) + b_2$, thì theo định lý đại số tuyến tính, tích của hai ma trận tuyến tính $W_2 W_1$ vẫn chỉ là MỘT ma trận tuyến tính duy nhất! Điều đó có nghĩa là dù bạn có xếp 100 tầng tuyến tính, mạng nơ-ron của bạn vẫn chỉ tương đương với một mô hình Hồi quy Tuyến tính đơn giản, hoàn toàn bất lực trước dữ liệu phi tuyến!

Để tạo ra "trí thông minh nhân tạo", chúng ta bắt buộc phải chèn các **Hàm kích hoạt phi tuyến (Non-linear Activation Functions)** vào giữa các tầng. Lớp `torch.nn.Module` là công cụ quyền năng nhất của PyTorch giúp bạn thiết kế, tùy biến và kiểm soát mọi kiến trúc mạng nơ-ron từ đơn giản đến những mô hình vĩ đại nhất thế giới.



## 2. CONCEPT — NỀN TẢNG LÝ THUYẾT & TRỰC GIÁC


### 1. Kiến trúc Perceptron Đa tầng (MLP)
Một mạng MLP chuẩn mực gồm 3 thành phần:
1. **Input Layer (Tầng đầu vào):** Nhận vector đặc trưng có $D_{in}$ chiều.
2. **Hidden Layers (Các tầng ẩn):** Nơi mạng nơ-ron học cách biến đổi và trừu tượng hóa đặc trưng. Mỗi tầng thực hiện: $h = \sigma(W x + b)$.
3. **Output Layer (Tầng đầu ra):** Đưa ra kết quả dự đoán với số chiều $D_{out}$ (1 nơ-ron cho hồi quy/nhị phân, $C$ nơ-ron cho phân loại đa lớp).

### 2. Tại sao hàm ReLU (Rectified Linear Unit) lại thống trị Deep Learning?
$$\text{ReLU}(z) = \max(0, z)$$
* **Giải quyết triệt tiêu đạo hàm (Vanishing Gradient):** Đạo hàm của ReLU luôn bằng 1 khi $z > 0$, giúp dòng gradient có thể lan truyền ngược qua hàng chục tầng mạng sâu mà không bị suy giảm về 0.
* **Tốc độ tính toán siêu nhanh:** Chỉ là phép so sánh lớn hơn 0 trên phần cứng CPU/GPU.

### 3. Cấu trúc chuẩn của một lớp `nn.Module`
* Trong `__init__(self)`: Khai báo các tầng mạng (`nn.Linear`, `nn.ReLU`, `nn.Dropout`).
* Trong `forward(self, x)`: Định nghĩa luồng dữ liệu di chuyển từ đầu vào xuyên qua các tầng để tạo ra đầu ra.



## 3. VISUAL INTUITION — TRỰC QUAN HÓA KIẾN THỨC


```
Cấu trúc Mạng nơ-ron Perceptron Đa tầng (MLP):
  x1 ──○
          x2 ──○ ──> [ Tầng ẩn 1: nn.Linear(3, 8) + ReLU ] ──> [ Tầng ẩn 2: nn.Linear(8, 4) + ReLU ] ──> [ Output: nn.Linear(4, 1) ] ──> y_hat
        /
  x3 ──○
```



## 4. CODE IMPLEMENTATION & 5. WALKTHROUGH

*Toàn bộ code dưới đây được thiết kế thực tế, có thể chạy độc lập không phụ thuộc tải dữ liệu bên ngoài.*

In [ ]:
import torch
import torch.nn as nn

# 1. Dinh nghia kien truc mang Perceptron Da tang (MLP) ke thua tu nn.Module
class MedicalClassifierMLP(nn.Module):
    def __init__(self, input_dim=4, hidden_dim=16, output_dim=1):
        super(MedicalClassifierMLP, self).__init__()
        
        # Tang an 1 (Hidden Layer 1)
        self.fc1 = nn.Linear(in_features=input_dim, out_features=hidden_dim)
        self.relu1 = nn.ReLU()
        
        # Tang an 2 (Hidden Layer 2)
        self.fc2 = nn.Linear(in_features=hidden_dim, out_features=hidden_dim // 2)
        self.relu2 = nn.ReLU()
        
        # Tang dau ra (Output Layer - Logits)
        self.out = nn.Linear(in_features=hidden_dim // 2, out_features=output_dim)
        
    def forward(self, x):
        # Dinh nghia luong chay (Forward Pass)
        x = self.fc1(x)
        x = self.relu1(x)
        x = self.fc2(x)
        x = self.relu2(x)
        logits = self.out(x)
        return logits

# Khoi tao mo hinh
model = MedicalClassifierMLP(input_dim=4, hidden_dim=16, output_dim=1)
print("===== KIEN TRUC MANG NƠ-RON (MODEL ARCHITECTURE) =====")
print(model)

### 🔍 Code Walkthrough (Khối 1)
Khối code thiết kế một mạng MLP hoàn chỉnh bằng `nn.Module`:
- Kế thừa chuẩn `super().__init__()`.
- Tách bạch 2 tầng ẩn có kích thước thu hẹp dần (4 -> 16 -> 8 -> 1) kết hợp phi tuyến `nn.ReLU()`.
- Tầng cuối cùng trả về `logits` thô (chưa qua Sigmoid) — chuẩn thiết kế tối ưu số học của PyTorch.


In [ ]:
# 2. Kiem tra Forward Pass, Debug kich thuoc Tensor va Dem so tham so
# Gia lap mot Mini-Batch gom 8 benh nhan, moi nguoi co 4 dac trung
dummy_batch = torch.randn(8, 4)

# Chay Forward Pass
with torch.no_grad():
    output_logits = model(dummy_batch)

print("===== KIEM THU FORWARD PASS =====")
print(f"Kich thuoc dau vao (Input Batch Shape)  : {dummy_batch.shape}")
print(f"Kich thuoc dau ra (Output Logits Shape): {output_logits.shape}")
print("\nGia tri 3 logits dau tien:\n", output_logits[:3])

# Dem tong so luong tham so co the huan luyen (Trainable Parameters)
total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"\nTong so tham so can huan luyen cua mang: {total_params} parameters")
for name, param in model.named_parameters():
    print(f" - {name:15s}: shape = {str(list(param.shape)):15s} ({param.numel()} params)")

### 🔍 Code Walkthrough (Khối 2)
Khối code kiểm tra tính đúng đắn của kiến trúc mạng:
- Đưa một batch giả lập $8 	imes 4$ qua mạng để xác minh kích thước đầu ra đúng chuẩn $(8, 1)$.
- Sử dụng hàm tổng hợp `numel()` duyệt qua `named_parameters()` để kiểm kê chi tiết số lượng trọng số (Weights) và độ lệch (Biases) của từng tầng.


## 6. PRACTICE — BÀI TẬP THỰC HÀNH


### Bài 1 (Easy): Xây dựng mạng bằng `nn.Sequential`
Tìm hiểu module đóng gói nhanh `nn.Sequential`. Viết lại mạng nơ-ron trên chỉ bằng một khối `nn.Sequential` duy nhất và kiểm tra xem đầu ra có giống hệt không.

### Bài 2 (Easy): Đổi sang bài toán Phân loại 3 lớp (Multi-class)
Điều chỉnh mô hình để giải quyết bài toán phân loại loài hoa Iris có 3 lớp (`output_dim=3`). Nêu rõ kích thước đầu ra thay đổi như thế nào khi đưa batch 8 mẫu vào mạng.

### Bài 3 (Medium): So sánh hàm kích hoạt ReLU vs LeakyReLU vs GELU
Thay thế `nn.ReLU()` bằng `nn.LeakyReLU(negative_slope=0.01)` và `nn.GELU()`. Giải thích tại sao LeakyReLU giúp giải quyết hiện tượng "Chết nơ-ron ReLU" (Dying ReLU Problem).

### Bài 4 (Medium): Khởi tạo trọng số tùy biến (Custom Weight Initialization)
Mặc định PyTorch sử dụng khởi tạo Kaiming/He hoặc Xavier. Hãy viết một hàm sử dụng `torch.nn.init.kaiming_normal_` để khởi tạo lại trọng số cho tất cả các tầng `nn.Linear` trong mô hình.

### Bài 5 (Challenge): Tự viết Custom Layer trừu tượng
Tạo một lớp kế thừa từ `nn.Module` tên là `LinearWithSkipConnection` kết hợp một tầng `nn.Linear` với một đường tắt thẳng (Residual/Skip connection) cộng trực tiếp đầu vào vào đầu ra: $y = 	ext{Linear}(x) + x$ (nền tảng của kiến trúc ResNet).



## 7. VOAI CONNECTION — CHIẾN LƯỢC PHÒNG THI OLYMPIC


Trong phòng thi Olympic AI, lỗi phổ biến nhất khiến thí sinh bế tắc hàng giờ là **Lỗi Lệch Kích Thước Tensor (Dimension Mismatch Error)**:
Ví dụ: Tầng Linear 1 có `out_features=32` nhưng tầng Linear 2 lại khai báo `in_features=16`, khiến code lập tức sập ngay ở forward pass đầu tiên. Thói quen luôn chạy thử một `dummy_batch` qua mạng để in shape của từng tầng là kỹ năng tự bảo vệ không thể thiếu của một đấu thủ chuyên nghiệp.



## 8. DEBUGGING CHALLENGE — BẮT LỖI THỰC CHIẾN


**Tình huống thực tế:** Thí sinh thêm hàm `nn.Sigmoid()` ở tầng cuối cùng của mạng nhưng lại sử dụng hàm mất mát `nn.BCEWithLogitsLoss()` trong quá trình huấn luyện:

```python
# CODE CÓ LỖI NHÂN ĐÔI HÀM SIGMOID:
# nn.BCEWithLogitsLoss() da tich hop san ham Sigmoid ben trong de on dinh so hoc (Numerical Stability)!
# Neu ban tu goi Sigmoid o tang cuoi, ham Sigmoid se bi ap dung 2 LAN LIEN TIEP,
# khien xac suat bi bop meo va gradient gan nhu bien mat!
```

**Nhiệm vụ của bạn:**
1. Giải thích sự khác biệt giữa `nn.BCELoss` (yêu cầu đầu vào đã qua Sigmoid) và `nn.BCEWithLogitsLoss` (yêu cầu đầu vào là Logits thô).
2. Nêu lý do tại sao các chuyên gia luôn khuyến nghị sử dụng `BCEWithLogitsLoss` kết hợp với tầng cuối không có activation.



## 9. KNOWLEDGE CHECK — KIỂM TRA ĐỘ HIỂU BẢN CHẤT


1. Điều gì sẽ xảy ra nếu ta xây dựng một mạng nơ-ron sâu gồm 10 tầng `nn.Linear` liên tiếp nhưng hoàn toàn KHÔNG sử dụng bất kỳ hàm kích hoạt phi tuyến nào?
   - A. Mạng sẽ đạt độ chính xác 100%.
   - B. Toàn bộ mạng 10 tầng sẽ suy biến toán học tương đương với đúng MỘT tầng tuyến tính duy nhất, mất khả năng học các mối quan hệ phi tuyến.
   - C. PyTorch sẽ tự động ngắt kết nối.
   - D. Mô hình sẽ chạy chậm hơn 100 lần.

2. Trong phương thức `forward(self, x)` của một lớp kế thừa `nn.Module`, tham số `x` đại diện cho điều gì?
   - A. Nhãn mục tiêu của bài toán.
   - B. Tensor dữ liệu đầu vào của Mini-batch hiện tại.
   - C. Giá trị đạo hàm của hàm mất mát.
   - D. Số lượng epochs đã hoàn thành.

3. Ưu điểm nổi bật nhất của hàm kích hoạt ReLU ($\max(0, z)$) so với Sigmoid trong các mạng nơ-ron sâu là gì?
   - A. ReLU có giá trị âm lớn.
   - B. ReLU không bị bão hòa ở miền dương (đạo hàm luôn bằng 1 khi $z > 0$), giúp triệt tiêu hiện tượng biến mất gradient (Vanishing Gradient).
   - C. ReLU chỉ chạy được trên CPU.
   - D. ReLU làm giảm kích thước dữ liệu.

4. Tổng số lượng tham số (Weights + Biases) của một tầng `nn.Linear(in_features=10, out_features=5)` là bao nhiêu?
   - A. $10 	imes 5 = 50$
   - B. $10 	imes 5 + 5 = 55$
   - C. $10 + 5 = 15$
   - D. $10 	imes 5 + 10 = 60$

5. Khái niệm "Logits" trong học sâu dùng để chỉ giá trị nào?
   - A. Giá trị hàm mất mát sau khi đạo hàm.
   - B. Giá trị thô chưa chuẩn hóa được xuất ra từ tầng tuyến tính cuối cùng của mạng trước khi đi qua hàm kích hoạt xác suất (như Sigmoid hay Softmax).
   - C. Số lượng nơ-ron bị tắt.
   - D. Giá trị tỷ lệ phần trăm chính xác.



## 10. SESSION SUMMARY — TỔNG KẾT BUỔI HỌC


* **What I learned:**
  - Kiến trúc Perceptron Đa tầng (MLP) và tầm quan trọng sống còn của Hàm kích hoạt phi tuyến.
  - Cách thiết kế mạng nơ-ron chuyên nghiệp bằng cách kế thừa `torch.nn.Module`.
  - Cơ chế đếm và kiểm soát tham số của từng tầng (`parameters()`, `named_parameters()`).
* **What I can now do:**
  - Tự tay lập trình và tùy biến bất kỳ kiến trúc mạng MLP nào cho bài toán hồi quy hoặc phân loại.
  - Debug kích thước Tensor qua từng tầng mạng bằng kỹ thuật `dummy_batch`.
* **What I still need to practice:**
  - Kết nối Dataset, DataLoader, Model và Optimizer vào một **Training Loop chuẩn mực** ở Buổi 4.



## 11. NEXT SESSION — BƯỚC ĐI TIẾP THEO


Trong **Buổi M3_S04**, chúng ta sẽ ghép nối tất cả các mảnh ghép lại: **Tự Viết Training Loop Chuẩn Mực Bằng PyTorch** — Làm chủ 5 bước thần thánh của một vòng lặp huấn luyện, phân biệt `model.train()` vs `model.eval()`, và theo dõi độ mất mát trên từng batch!



---

## 🎯 Syllabus Traceability (Đối chiếu Syllabus)

Buổi học này đóng góp trực tiếp vào mục tiêu của **M3**:
- Kiến trúc MLP & Non-linear activations (ReLU, Sigmoid) ✓
- Kế thừa và thiết kế mạng nơ-ron bằng torch.nn.Module ✓
- Forward pass, theo dõi shape tensor & quản lý parameters ✓

* **Session Deliverable:** Mã nguồn định nghĩa kiến trúc mạng nơ-ron MLP hoàn chỉnh bằng nn.Module
* **Đảm bảo chuẩn Syllabus Lock:** Nội dung nằm đúng chuẩn 100%, không lan man, giải quyết trực tiếp năng lực cốt lõi.
